# Denoising — tensor interface

Call `antstorch.denoise_image` with a tensor rather than an ANTsImage. The same
real r16 input and Rician denoising parameters are used by
[the ANTsImage example](denoising_antsimage.ipynb). No pretrained weights are needed.
The returned tensor retains the tensor shape, dtype and device; physical metadata
must be retained separately when converting back to an image.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
from time import perf_counter
_example_started = perf_counter()

import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

# Synchronize the selected accelerator so timings include completed GPU work.
from contextlib import contextmanager
_example_operation_times = []

def _synchronize_example_device():
    selected = torch.device(device)
    if selected.type == "cuda":
        torch.cuda.synchronize(selected)
    elif selected.type == "mps":
        torch.mps.synchronize()

@contextmanager
def time_step(label):
    _synchronize_example_device()
    started = perf_counter()
    try:
        yield
    finally:
        _synchronize_example_device()
        seconds = perf_counter() - started
        _example_operation_times.append((label, seconds))
        print(f"{label}: {seconds:.4f} s")

## Keep tensor axes and voxel spacing in the same order
Here the tensor is `(1, 1, y, x)`, so the spacing argument is `(spacing_y,
spacing_x)`. Denoising associates spacing with the supplied array axes; unlike
N4's `ImageDomain`, it does not take an ITK-order domain object. Reverse the
image spacing when reversing its array axes. The mask follows the same layout.

In [ ]:
from antstorch.syn.bridge import ants_image_to_tensor, tensor_to_ants_image

reference = ants.image_read(ants.get_ants_data("r16")).clone("float")
reference = ants.resample_image(reference, (128, 96), use_voxels=True)
reference_mask = ants.get_mask(reference)
image = ants_image_to_tensor(reference, device, normalize=False)
mask = ants_image_to_tensor(reference_mask, device, normalize=False)
spacing_in_tensor_order = tuple(reversed(reference.spacing))
assert image.shape == (1, 1, *reversed(reference.shape))
with time_step("Denoising — tensor"):
    output = antstorch.denoise_image(
        image, mask=mask, spacing=spacing_in_tensor_order,
        noise_model="Rician", p=1, r=2, shrink_factor=1, device=device,
    )
assert torch.is_tensor(output) and output.shape == image.shape
assert output.dtype == image.dtype and output.device == image.device
assert torch.isfinite(output).all()
denoised = tensor_to_ants_image(output, reference)
assert ants.image_physical_space_consistency(denoised, reference)

In [ ]:
original = reference.numpy()
filtered = denoised.numpy()
residual = original - filtered
fig, axes = plt.subplots(1, 3, figsize=(10, 3), constrained_layout=True)
for ax, array, title in zip(axes, [original, filtered, residual],
                            ["Tutorial r16", "Tensor denoising", "Removed residual"]):
    view = ax.imshow(array.T, origin="lower", cmap="gray",
                     vmin=None if title == "Removed residual" else original.min(),
                     vmax=None if title == "Removed residual" else original.max())
    ax.set_title(title); ax.axis("off")
    fig.colorbar(view, ax=ax, shrink=0.7)
plt.show()

Intensity and mask tensors carry no origin or direction metadata. The original
reference image restores them for display or export. The residual is not a
noise ground truth: inspect edges and small structures after filtering.

## Execution time

Run **Restart Kernel and Run All Cells** for a comparable end-to-end duration.
The total starts at the first setup cell and includes imports, downloads, data
preparation, computation, plotting and file writes. Kernel startup is excluded;
manual pauses between cells are included. Operation timings include any
preprocessing or asset downloads performed inside the called function. First
runs can be slower because caches are empty. Timings describe this run on the
selected device, not a fixed performance estimate for other machines.

In [ ]:
_synchronize_example_device()
total_seconds = perf_counter() - _example_started
operation_seconds = sum(seconds for _, seconds in _example_operation_times)
print(f"Total elapsed time: {total_seconds:.2f} s ({total_seconds / 60:.2f} min)")
print(f"Timed operations: {operation_seconds:.4f} s on {device}")